In [102]:
import zipfile
import re
from lxml import etree
from tqdm import tqdm

def extract_year_from_text(root):
    """
    Extracts the historical publication year by looking specifically 
    inside the 'sourceDesc' (Source Description) block.
    """
    path = ".//*[local-name()='sourceDesc']//*[local-name()='date' and @type='publication']"
    date_elements = root.xpath(path)
    
    if date_elements and date_elements[0].text:
        match = re.search(r'\d{4}', date_elements[0].text)
        if match:
            return match.group()
            
    return "0000"

def process_zip_directly(zip_path, output_tsv):
    with zipfile.ZipFile(zip_path, 'r') as z:
        # ONLY grab files that are inside a 'full' directory
        file_list = [f for f in z.namelist() if f.endswith(('.tcf', '.xml')) and '/full/' in f]
        
        print(f"Found {len(file_list)} files in the 'full' directory.")
        
        with open(output_tsv, "w", encoding="utf-8") as out:
            for file_name in tqdm(file_list, desc="Processing TCF (Full)"):
                with z.open(file_name) as f:
                    try:
                        tree = etree.parse(f)
                        root = tree.getroot()
                        
                        year = extract_year_from_text(root)

                        # Create Lookup Maps using local-name() to safely ignore XML prefix headaches
                        tokens = {t.get("ID"): (t.text or "") for t in root.xpath(".//*[local-name()='token']")}
                        lemmas = {l.get("tokenIDs"): (l.text or "") for l in root.xpath(".//*[local-name()='lemma']")}
                        pos_tags = {p.get("tokenIDs"): (p.text or "") for p in root.xpath(".//*[local-name()='tag']")}

                        # Skip if the file is completely empty/broken to avoid messing up data.py
                        if not tokens:
                            continue

                        # Write the <sod> marker with the filename so your data.py tracks duplicates
                        out.write(f"<sod>\t{file_name}\t{year}\n")

                        # Reconstruct Sentences
                        for sent in root.xpath(".//*[local-name()='sentence']"):
                            t_ids = sent.get("tokenIDs", "").split()
                            for tid in t_ids:
                                word = tokens.get(tid, "")
                                lemma = lemmas.get(tid, word) 
                                pos = pos_tags.get(tid, "UNK")
                                
                                # 3 columns for your data.py
                                out.write(f"{word}\t{lemma}\t{pos}\n")
                            
                            # 3 columns for EOS
                            out.write("<eos>\t<eos>\t<eos>\n")
                            
                    except Exception as e:
                        print(f"Skipping {file_name} due to error: {e}")


In [104]:
if __name__ == "__main__":
    ZIP_PATH = "dta_kernkorpus_2026-02-10_tcf.zip"
    OUTPUT_TSV = "dta_corpus.tsv"
    process_zip_directly(ZIP_PATH, OUTPUT_TSV)

Found 1473 files in the 'full' directory.


Processing TCF (Full):  55%|█████▍    | 810/1473 [31:16<26:47,  2.42s/it]  

Skipping dta_kernkorpus_2026-02-10/full/arnold_ketzerhistorie02_1700.tcf.xml due to error: Error in xpath expression


Processing TCF (Full): 100%|██████████| 1473/1473 [55:51<00:00,  2.28s/it] 


In [105]:
with open("dta_corpus.tsv", encoding="utf-8") as f:
    for _ in range(20):
        print(f.readline())

<sod>	dta_kernkorpus_2026-02-10/full/meissner_krimi_1796.tcf.xml	1796

Kriminal	Kriminal	NN

GESCHICHTEN	Geschichte	NN

von	von	APPR

A	A	NE

G	G	NE

Meißner	Meißner	ADJA

.	.	$.

Wien	Wien	NE

1796	1796	CARD

.	.	$.

<eos>	<eos>	<eos>

I.	i.	ADJA

Mord	Mord	NN

an	an	APPR

ſeiner	seine	PPOSAT

Frau	Frau	NN

,	,	$,

um	um	KOUI

ihre	ihr	PPOSAT



In [107]:
import json
from collections import defaultdict

# Import the specific time slices and delimiter from their codebase
from util import COARSE_TIME_SLICES, SPACE_REPLACEMENT_IN_FILENAMES

def build_2_grams_from_tsv(tsv_path, output_json, corpus_type="DTA"):
    # 1. Setup the Time Slices
    time_slices = COARSE_TIME_SLICES[corpus_type]
    num_slices = len(time_slices)
    
    def get_slice_index(year):
        """Finds which bucket the year belongs to."""
        for i, (start, end) in enumerate(time_slices):
            if start <= year <= end:
                return i
        return None

    # 2. Initialize Dictionary
    bigram_counts = defaultdict(lambda: [0] * num_slices)
    seen_docs = set()
    
    current_slice_idx = None
    current_lemmas = []
    skip_doc = False

    # 3. Read directly from the TSV file
    print(f"Reading directly from {tsv_path}...")
    with open(tsv_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue

            parts = line.split('\t')
            if len(parts) != 3: continue
            tok, lem, tag = parts

            # Document Start
            if tok == "<sod>":
                doc_id = lem # Column 2 is the filename
                if doc_id in seen_docs:
                    skip_doc = True
                else:
                    skip_doc = False
                    seen_docs.add(doc_id)
                    try:
                        year = int(tag)
                        current_slice_idx = get_slice_index(year)
                    except ValueError:
                        current_slice_idx = None
                        skip_doc = True # Skip if year is missing/broken
                        
            # End of Sentence
            elif tok == "<eos>":
                if not skip_doc and current_slice_idx is not None and len(current_lemmas) >= 2:
                    # Slide a window of size 2 across the sentence lemmas
                    for i in range(len(current_lemmas) - 1):
                        w1 = current_lemmas[i].lower()
                        w2 = current_lemmas[i+1].lower()
                        
                        # Glue them together using the researchers' special delimiter
                        bg_key = f"{w1}{SPACE_REPLACEMENT_IN_FILENAMES}{w2}"
                        
                        # Add +1 to the specific historical time slice index
                        bigram_counts[bg_key][current_slice_idx] += 1
                        
                # Clear the sentence buffer for the next sentence
                current_lemmas = []
                
            # Standard word
            elif not skip_doc:
                # We use the lemma (column 2) to track semantic shifts
                current_lemmas.append(lem)

    # 4. Save to JSON
    print("Writing to JSON...")
    with open(output_json, 'w', encoding='utf-8') as out_f:
        json.dump(bigram_counts, out_f, ensure_ascii=False)
        
    print(f"Success! Saved {len(bigram_counts)} unique bigrams to {output_json}")

if __name__ == "__main__":
    TSV_PATH = "dta_corpus.tsv" 
    OUTPUT_JSON = "DTA_2_grams.json" 
    
    build_2_grams_from_tsv(TSV_PATH, OUTPUT_JSON)

Reading directly from dta_corpus.tsv...
Writing to JSON...
Success! Saved 7720889 unique bigrams to DTA_2_grams.json


In [2]:
import pickle
from transformers import AutoTokenizer

# --- Configuration ---
TSV_FILE = "dta_corpus.tsv"
OUTPUT_PKL = "extracted_targets.pkl"
MODEL_NAME = "dbmdz/bert-base-german-cased"

# The 43 German targets from Table 8 of the paper
TARGET_COMPOUNDS = {
    "ruhestand", 
}

class Sentence:
    def __init__(self, tokens, lemmas, tags, year):
        self.tokens = tokens
        self.lemmas = lemmas
        self.tags = tags
        self.year = year
        
    def __repr__(self):
        return f"[{self.year}] {' '.join(self.tokens)}"

def extract_and_pickle():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    extracted_data = []
    seen_docs = set()
    
    current_tokens, current_lemmas, current_tags = [], [], []
    current_year = None
    skip_doc = False

    print("Scanning TSV for target compounds...")
    with open(TSV_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 3:
                continue
            
            tok, lem, tag = parts
            
            if tok == "<sod>":
                doc_id = lem
                if doc_id in seen_docs:
                    skip_doc = True
                else:
                    skip_doc = False
                    seen_docs.add(doc_id)
                    try:
                        current_year = int(tag)
                    except ValueError:
                        skip_doc = True
                        
            elif tok == "<eos>":
                if not skip_doc and current_lemmas:
                    # Check if any lemma in the sentence is in our target list
                    # We lower() the lemma to ensure it matches our set
                    sentence_lemmas_lower = [l.lower() for l in current_lemmas]
                    
                    if any(target in sentence_lemmas_lower for target in TARGET_COMPOUNDS):
                        
                        # Create the Sentence object
                        sent_obj = Sentence(
                            list(current_tokens), 
                            list(current_lemmas), 
                            list(current_tags), 
                            current_year
                        )
                        
                        # Calculate the exact tokenizer spans
                        current_pos = 1 # Start at 1 to account for BERT's [CLS] token
                        
                        for i, lemma in enumerate(current_lemmas):
                            sub_tokens = tokenizer(lemma, add_special_tokens=False)["input_ids"]
                            if not sub_tokens:
                                sub_tokens = [tokenizer.unk_token_id]
                                
                            start_idx = current_pos
                            end_idx = current_pos + len(sub_tokens)
                            
                            # If this specific word is one of our targets, save the dictionary!
                            if lemma.lower() in TARGET_COMPOUNDS:
                                extracted_data.append({
                                    "sent": sent_obj,
                                    "lemma-span": (start_idx, end_idx),
                                    "target_lemma": lemma.lower()
                                })
                                
                            current_pos = end_idx

                # Clear buffers for the next sentence
                current_tokens.clear()
                current_lemmas.clear()
                current_tags.clear()
                
            elif not skip_doc:
                current_tokens.append(tok)
                current_lemmas.append(lem)
                current_tags.append(tag)

    print(f"Found {len(extracted_data)} target occurrences. Pickling...")
    
    with open(OUTPUT_PKL, "wb") as out_file:
        # We can just dump the whole list at once since it's filtered and small
        pickle.dump(extracted_data, out_file, protocol=pickle.HIGHEST_PROTOCOL)
        
    print(f"Done! Saved to {OUTPUT_PKL}")

if __name__ == "__main__":
    extract_and_pickle()

Scanning TSV for target compounds...
Found 933 target occurrences. Pickling...
Done! Saved to extracted_targets.pkl


In [38]:
import csv
import pickle
import os
from transformers import AutoTokenizer

# --- Configuration ---
TSV_FILE = "dta_corpus.tsv"
OUTPUT_DIR = "german_target_pickles"           # Folder to store the 43 .pkl files
COUNTS_CSV = "german_target_counts.csv"        # The summary table
MODEL_NAME = "dbmdz/bert-base-german-cased"

# The 43 German targets from Table 8
TARGET_COMPOUNDS = {
    "ruhestand", "rechtsstreit", "uhrwerk", "triebwerk", "murmeltier", 
    "eisenwerk", "trauerspiel", "stückwerk", "streitsache", "zeughaus", 
    "gesichtszug", "feuerwerk", "kartenspiel", "wortspiel", "schauspiel", 
    "hausstand", "grundfläche", "meerwasser", "eifersucht", "sündenfall", 
    "sonnenuhr", "tagewerk", "mauerwerk", "sonnenstrahl", "heerführer", 
    "sonnenlicht", "bergwerk", "ziegenbock", "sonnenschein", "kreuzzug", 
    "brunnenwasser", "seewasser", "zitronensaft", "stockwerk", "kinderspiel", 
    "wunderwerk", "bildhauer", "leinöl", "rehbock", "windspiel", 
    "sonnenblume", "salzwasser", "feldzug"
}

# MUST be defined so pickle knows how to rebuild the objects
class Sentence:
    def __init__(self, sentence_id, tokens, lemmas, tags, year):
        self.sentence_id = sentence_id  # NEW: Store the ID
        self.tokens = tokens
        self.lemmas = lemmas
        self.tags = tags
        self.year = year
        
    def __repr__(self):
        # NEW: Print the ID in the string representation
        return f"[ID: {self.sentence_id} | {self.year}] {' '.join(self.tokens)}"

def extract_and_split():
    # 1. Setup
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    # Initialize a dictionary to hold lists for each target
    target_data = {target: [] for target in TARGET_COMPOUNDS}
    seen_docs = set()
    
    current_tokens, current_lemmas, current_tags = [], [], []
    current_year = None
    skip_doc = False
    
    global_sent_id = 0  # NEW: Track absolute position in the TSV

    print(f"Scanning {TSV_FILE} and calculating spans...")
    
    # 2. Process the TSV
    with open(TSV_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 3:
                continue
            
            tok, lem, tag = parts
            
            if tok == "<sod>":
                doc_id = lem
                if doc_id in seen_docs:
                    skip_doc = True
                else:
                    skip_doc = False
                    seen_docs.add(doc_id)
                    try:
                        current_year = int(tag)
                    except ValueError:
                        skip_doc = True
                        
            elif tok == "<eos>":
                if not skip_doc and current_lemmas:
                    global_sent_id += 1 # NEW: Increment ID for every valid sentence in the corpus
                    
                    sentence_lemmas_lower = [l.lower() for l in current_lemmas]
                    
                    # Find which targets are in this sentence (could be more than one!)
                    found_targets = [t for t in TARGET_COMPOUNDS if t in sentence_lemmas_lower]
                    
                    if found_targets:
                        sent_obj = Sentence(
                            global_sent_id, # NEW: Pass the ID to the object
                            list(current_tokens), 
                            list(current_lemmas), 
                            list(current_tags), 
                            current_year
                        )
                        
                        # Calculate the tokenizer spans for the whole sentence
                        current_pos = 1 # Start at 1 for the [CLS] token
                        spans = {}      # Map lemma index to its (start, end) span
                        
                        for i, lemma in enumerate(current_lemmas):
                            sub_tokens = tokenizer(lemma, add_special_tokens=False)["input_ids"]
                            if not sub_tokens:
                                sub_tokens = [tokenizer.unk_token_id]
                                
                            start_idx = current_pos
                            end_idx = current_pos + len(sub_tokens)
                            spans[i] = (start_idx, end_idx)
                            current_pos = end_idx
                            
                        # Assign the sentence and specific span to the correct target buckets
                        for i, lemma in enumerate(sentence_lemmas_lower):
                            if lemma in found_targets:
                                target_data[lemma].append({
                                    "sent": sent_obj,
                                    "lemma-span": spans[i],
                                    "target_lemma": lemma
                                })

                # Clear buffers
                current_tokens.clear()
                current_lemmas.clear()
                current_tags.clear()
                
            elif not skip_doc:
                current_tokens.append(tok)
                current_lemmas.append(lem)
                current_tags.append(tag)

    # 3. Export Pickles and write the Summary CSV
    print(f"Writing individual pickles and {COUNTS_CSV}...")
    
    with open(COUNTS_CSV, 'w', encoding='utf-8', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Target Compound", "Sentence Count"])
        
        for target, data_list in target_data.items():
            count = len(data_list)
            writer.writerow([target, count])
            
            # Only generate a .pkl if the word actually appeared in the corpus
            if count > 0:
                pkl_path = os.path.join(OUTPUT_DIR, f"{target}.pkl")
                with open(pkl_path, "wb") as out_file:
                    pickle.dump(data_list, out_file, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Done! All data securely prepped and saved.")

if __name__ == "__main__":
    extract_and_split()

Scanning dta_corpus.tsv and calculating spans...
Writing individual pickles and german_target_counts.csv...
Done! All data securely prepped and saved.


In [41]:
import pickle

# 1. We MUST redefine the Sentence class here so pickle knows how to rebuild the objects!
class Sentence:
    def __init__(self, tokens, lemmas, tags, year):
        self.tokens = tokens
        self.lemmas = lemmas
        self.tags = tags
        self.year = year
        
    def __repr__(self):
        return f"[{self.year}] {' '.join(self.tokens)}"

def inspect_data(filepath="german_target_pickles/bergwerk.pkl"):
    print(f"Opening {filepath}...\n")
    
    # 2. Load the binary file
    with open(filepath, "rb") as f:
        data = pickle.load(f)
        
    # 3. Print high-level stats
    print(f"Data Type: {type(data)}")
    print(f"Total target occurrences extracted: {len(data)}\n")
    
    # 4. Deep dive into the first two entries
    if len(data) > 0:
        for i in range(min(5, len(data))): # Change '2' to see more entries
            item = data[i]
            sent_obj = item["sent"]
            
            print(f"--- ENTRY {i + 1} ---")
            print(f"Target Lemma: '{item['target_lemma']}'")
            print(f"Lemma Span:   {item['lemma-span']}")
            print(f"Year:         {sent_obj.year}")
            print(f"ID:         {sent_obj.sentence_id}")
            
            # Print the actual text
            print(f"Raw Text:     {sent_obj}")
            
            # Show how to access the internal lists
            print(f"Tokens List:  {sent_obj.tokens[:10]}...") # Printing just the first 10 for readability
            print(f"Lemmas List:  {sent_obj.lemmas}...")
            print(f"Tags List:    {sent_obj.tags[:10]}...\n")

if __name__ == "__main__":
    inspect_data()

Opening german_target_pickles/bergwerk.pkl...

Data Type: <class 'list'>
Total target occurrences extracted: 2248

--- ENTRY 1 ---
Target Lemma: 'bergwerk'
Lemma Span:   (12, 14)
Year:         1890
ID:         14433
Raw Text:     [1890] Schon 3000 v. Chr. erkämpften sie sich die Bergwerke am Sinai , und der dort sich entwickelnde Bergbau fand eine ungemeine Unterstützung in der Neigung des Arabers zur Handelsthätigkeit .
Tokens List:  ['Schon', '3000', 'v.', 'Chr.', 'erkämpften', 'sie', 'sich', 'die', 'Bergwerke', 'am']...
Lemmas List:  ['schon', '3000', 'v.', 'Chr.', 'erkämpft', 'sie', 'sich', 'd', 'Bergwerk', 'am', 'Sinai', ',', 'und', 'd', 'dort', 'sich', 'entwickelnd', 'Bergbau', 'finden', 'eine', 'ungemein', 'Unterstützung', 'in', 'd', 'Neigung', 'd', 'Araber', 'zur', 'Handelstätigkeit', '.']...
Tags List:    ['ADV', 'CARD', 'APPRART', 'NN', 'ADJA', 'PPER', 'PRF', 'ART', 'NN', 'APPRART']...

--- ENTRY 2 ---
Target Lemma: 'bergwerk'
Lemma Span:   (1, 3)
Year:         1868
ID:      

In [21]:
import pandas as pd
import random
import pickle
import os
import sys
import csv
import re

# 1. Custom Unpickler to bypass the "No module named 'data'" error
class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'data':
            if not hasattr(sys.modules[__name__], name):
                setattr(sys.modules[__name__], name, type(name, (object,), {}))
            return getattr(sys.modules[__name__], name)
        return super().find_class(module, name)

def generate_english_annotation_samples(directory_path="merged_full_sent", output_file="english_annotation_tasks.csv"):
    all_pairs = []
    
    # Study-defined eras for the CCOHA corpus 
    EARLY_RANGE = (1830, 1859) 
    LATE_RANGE = (1980, 2009) 
    
    # Helper to extract the year from the "[YYYY] sentence..." format in 'sent'
    def get_year(item):
        sent = item.get('sent', '') if isinstance(item, dict) else getattr(item, 'sent', '')
        # FORCE string conversion here to handle custom token objects
        match = re.search(r'\[(\d{4})\]', str(sent))
        return int(match.group(1)) if match else None

    # Helper to get the clean sentence from 'raw_sentence'
    def get_text_content(item):
        raw_sent = item.get('raw_sentence', '') if isinstance(item, dict) else getattr(item, 'raw_sentence', '')
        # FORCE string conversion and strip any leading/trailing whitespace
        return str(raw_sent).strip()

    if not os.path.exists(directory_path):
        print(f"Error: Directory '{directory_path}' not found.")
        return

    for filename in os.listdir(directory_path):
        if filename.endswith(".pickle"):
            target_name = filename.replace(".pickle", "").replace("🔥", "").strip()
            file_path = os.path.join(directory_path, filename)
            
            try:
                with open(file_path, 'rb') as f:
                    instances = SafeUnpickler(f).load()
            except Exception as e:
                print(f"Failed to load {filename}: {e}")
                continue
            
            # 3. Filter instances by the required eras using the regex extractor
            early_instances = [i for i in instances if get_year(i) is not None and EARLY_RANGE[0] <= get_year(i) <= EARLY_RANGE[1]]
            late_instances = [i for i in instances if get_year(i) is not None and LATE_RANGE[0] <= get_year(i) <= LATE_RANGE[1]]
            
            if len(early_instances) < 2 or len(late_instances) < 2:
                print(f"Skipping '{target_name}': Insufficient data (Early: {len(early_instances)}, Late: {len(late_instances)})")
                continue

            target_pairs = []
            
            # The study used ~393 total English sentence pairs across 19 targets
            # 4. Create the 3 specific groups of sentence pairs 
            
            for _ in range(7):
                s1, s2 = random.sample(early_instances, 2)
                target_pairs.append([target_name, get_text_content(s1), get_text_content(s2), "Early-Early"])

            for _ in range(7):
                s1, s2 = random.sample(late_instances, 2)
                target_pairs.append([target_name, get_text_content(s1), get_text_content(s2), "Late-Late"])

            for _ in range(7):
                s1 = random.choice(early_instances)
                s2 = random.choice(late_instances)
                target_pairs.append([target_name, get_text_content(s1), get_text_content(s2), "Cross-Era"])
            
            all_pairs.extend(target_pairs)

    random.shuffle(all_pairs)
    
    # 5. Export to CSV
    if all_pairs:
        df = pd.DataFrame(all_pairs, columns=['target_word', 'sentence_1', 'sentence_2', 'era_group'])
        df.to_csv(output_file, index=False, quoting=csv.QUOTE_ALL)
        print(f"\nSuccess! Generated {len(df)} sentence pairs and saved to '{output_file}'.")
    else:
        print("\nNo pairs were generated. Check your data structure or era filters.")

if __name__ == "__main__":
    generate_english_annotation_samples('merged_full_sent')

Skipping 'balancesheet': Insufficient data (Early: 1, Late: 9)
Skipping 'bankaccount': Insufficient data (Early: 0, Late: 21)
Skipping 'calendarmonth': Insufficient data (Early: 0, Late: 1)
Skipping 'elbowroom': Insufficient data (Early: 2, Late: 0)
Skipping 'fairytale': Insufficient data (Early: 1, Late: 19)
Skipping 'fieldwork': Insufficient data (Early: 1, Late: 4)
Skipping 'footsoldier': Insufficient data (Early: 0, Late: 3)
Skipping 'ins.company': Insufficient data (Early: 0, Late: 3)
Skipping 'lovesong': Insufficient data (Early: 1, Late: 5)
Skipping 'nestegg': Insufficient data (Early: 0, Late: 3)
Skipping 'silverspoon': Insufficient data (Early: 1, Late: 2)
Skipping 'wintersolstice': Insufficient data (Early: 1, Late: 2)

Success! Generated 147 sentence pairs and saved to 'english_annotation_tasks.csv'.


In [40]:
import pandas as pd
import random
import pickle
import os
import sys
import csv

# 1. Custom Unpickler to bypass the "No module named 'data'" error
class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'data':
            if not hasattr(sys.modules[__name__], name):
                setattr(sys.modules[__name__], name, type(name, (object,), {}))
            return getattr(sys.modules[__name__], name)
        return super().find_class(module, name)

def generate_german_annotation_samples(directory_path="target_pickles", output_file="german_annotation_tasks.csv", seed=42):
    # Set the random seed for perfect reproducibility!
    random.seed(seed)
    
    all_pairs = []
    
    # Study-defined eras for the DTA corpus 
    EARLY_RANGE = (1700, 1759)
    LATE_RANGE = (1870, 1909)
    
    # --- UPGRADED HELPER FUNCTIONS ---
    # These now access the object attributes directly instead of using regex!
    def get_obj(item):
        return item.get('sent') if isinstance(item, dict) else getattr(item, 'sent', item)

    def get_year(item):
        return getattr(get_obj(item), 'year', None)

    def get_id(item):
        return getattr(get_obj(item), 'sentence_id', 'N/A')

    def get_text_content(item):
        obj = get_obj(item)
        if hasattr(obj, 'tokens'):
            return " ".join(obj.tokens)
        return str(obj) # Fallback

    if not os.path.exists(directory_path):
        print(f"Error: Directory '{directory_path}' not found.")
        return

    for filename in os.listdir(directory_path):
        if filename.endswith(".pkl"):
            target_name = filename.replace(".pkl", "").strip()
            file_path = os.path.join(directory_path, filename)
            
            try:
                with open(file_path, 'rb') as f:
                    instances = SafeUnpickler(f).load()
            except Exception as e:
                print(f"Failed to load {filename}: {e}")
                continue
            
            # 3. Filter instances by the required DTA eras
            early_instances = [i for i in instances if get_year(i) is not None and EARLY_RANGE[0] <= get_year(i) <= EARLY_RANGE[1]]
            late_instances = [i for i in instances if get_year(i) is not None and LATE_RANGE[0] <= get_year(i) <= LATE_RANGE[1]]
            
            if len(early_instances) < 2 or len(late_instances) < 2:
                print(f"Skipping '{target_name}': Insufficient data (Early: {len(early_instances)}, Late: {len(late_instances)})")
                continue

            target_pairs = []
            
            # 4. Create the 3 specific groups of sentence pairs (21 total per word)
            # Now appending the IDs alongside the text
            for _ in range(7):
                s1, s2 = random.sample(early_instances, 2)
                target_pairs.append([
                    target_name, get_id(s1), get_text_content(s1), get_id(s2), get_text_content(s2), "Early-Early"
                ])

            for _ in range(7):
                s1, s2 = random.sample(late_instances, 2)
                target_pairs.append([
                    target_name, get_id(s1), get_text_content(s1), get_id(s2), get_text_content(s2), "Late-Late"
                ])

            for _ in range(7):
                s1 = random.choice(early_instances)
                s2 = random.choice(late_instances)
                target_pairs.append([
                    target_name, get_id(s1), get_text_content(s1), get_id(s2), get_text_content(s2), "Cross-Era"
                ])
            
            all_pairs.extend(target_pairs)

    # Shuffle the final list so annotators don't get all the same words/eras in a row
    random.shuffle(all_pairs)
    
    # 5. Export to CSV
    if all_pairs:
        # Added the new ID columns to the DataFrame
        df = pd.DataFrame(all_pairs, columns=[
            'target_word', 'sentence_1_id', 'sentence_1', 'sentence_2_id', 'sentence_2', 'era_group'
        ])
        df.to_csv(output_file, index=False, quoting=csv.QUOTE_ALL)
        print(f"\nSuccess! Generated {len(df)} sentence pairs and saved to '{output_file}'.")
    else:
        print("\nNo pairs were generated. Check your data structure or era filters.")

if __name__ == "__main__":
    generate_german_annotation_samples('german_target_pickles')


Success! Generated 903 sentence pairs and saved to 'german_annotation_tasks.csv'.


In [35]:
# count_targets.py

# --- Configuration ---
TSV_FILE = "dta_corpus.tsv"

# The 43 German targets from Table 8 of the paper
TARGET_COMPOUNDS = {
    "ruhestand", "rechtsstreit", "uhrwerk", "triebwerk", "murmeltier", 
    "eisenwerk", "trauerspiel", "stückwerk", "streitsache", "zeughaus", 
    "gesichtszug", "feuerwerk", "kartenspiel", "wortspiel", "schauspiel", 
    "hausstand", "grundfläche", "meerwasser", "eifersucht", "sündenfall", 
    "sonnenuhr", "tagewerk", "mauerwerk", "sonnenstrahl", "heerführer", 
    "sonnenlicht", "bergwerk", "ziegenbock", "sonnenschein", "kreuzzug", 
    "brunnenwasser", "seewasser", "zitronensaft", "stockwerk", "kinderspiel", 
    "wunderwerk", "bildhauer", "leinöl", "rehbock", "windspiel", 
    "sonnenblume", "salzwasser", "feldzug"
}

def count_target_sentences():
    # Initialize a dictionary to keep track of the counts
    counts = {target: 0 for target in TARGET_COMPOUNDS}
    current_lemmas = []
    total_sentences_scanned = 0
    
    print(f"Scanning {TSV_FILE} to count target occurrences. This will be quick...")
    
    with open(TSV_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 3:
                continue
            
            tok, lem, tag = parts
            
            if tok == "<sod>":
                continue # Skip document markers
                
            elif tok == "<eos>":
                if current_lemmas:
                    total_sentences_scanned += 1
                    
                    # Convert sentence lemmas to a lowercase set for lightning-fast matching
                    sentence_lemmas_lower = set(l.lower() for l in current_lemmas)
                    
                    # Find exactly which targets are in this sentence using set intersection
                    found_targets = TARGET_COMPOUNDS.intersection(sentence_lemmas_lower)
                    
                    # Increment the counter for each found target
                    for target in found_targets:
                        counts[target] += 1
                        
                # Clear buffer for the next sentence
                current_lemmas.clear()
                
            else:
                # We only need to track the lemmas to match our target list
                current_lemmas.append(lem)

    # Sort the results from highest frequency to lowest
    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    
    # Print the final report
    print(f"\nFinished scanning {total_sentences_scanned:,} sentences.")
    print("-" * 40)
    print(f"{'Target Compound':<20} | {'Sentence Count'}")
    print("-" * 40)
    
    for target, count in sorted_counts:
        print(f"{target:<20} | {count:,}")
        
    # Optional: Calculate total extracted sentences
    total_extracted = sum(count for count in counts.values())
    print("-" * 40)
    print(f"Total target instances found: {total_extracted:,}")

if __name__ == "__main__":
    count_target_sentences()

Scanning dta_corpus.tsv to count target occurrences. This will be quick...

Finished scanning 6,136,396 sentences.
----------------------------------------
Target Compound      | Sentence Count
----------------------------------------
schauspiel           | 3,188
bergwerk             | 2,118
feldzug              | 1,970
sonnenschein         | 1,801
eifersucht           | 1,624
sonnenstrahl         | 1,265
bildhauer            | 1,162
stockwerk            | 1,137
wunderwerk           | 1,079
trauerspiel          | 1,075
eisenwerk            | 1,037
ruhestand            | 928
mauerwerk            | 832
sonnenlicht          | 724
rechtsstreit         | 694
salzwasser           | 652
feuerwerk            | 605
grundfläche          | 593
gesichtszug          | 506
heerführer           | 468
zeughaus             | 419
kreuzzug             | 380
uhrwerk              | 374
brunnenwasser        | 284
tagewerk             | 280
meerwasser           | 262
kinderspiel          | 241
hausstand     